## Read all the files from Volume

In [0]:
from pyspark.sql.functions import *
# Bronze Parquet paths
customers_path = "/Volumes/ecommerce_catalog/bronze/bronze_files/customers"
products_path = "/Volumes/ecommerce_catalog/bronze/bronze_files/products"
orders_path = "/Volumes/ecommerce_catalog/bronze/bronze_files/orders"

# Read Bronze Parquet files
customers_df = spark.read.parquet(customers_path)
products_df = spark.read.parquet(products_path)
orders_df = spark.read.parquet(orders_path)

# Display the data
display(customers_df)
display(products_df)
display(orders_df)

## CUSTOMER_DATAQUALITY

In [0]:

print("Total customer rows:", customers_df.count())

print("Duplicate customer IDs:")
customers_df.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null customer IDs:")
customers_df.filter(col("customer_id").isNull()).show()

print("Null customer names:")
customers_df.filter(col("customer_name").isNull()).show()

## CLEANING CUSTOMERS DATA

In [0]:
from pyspark.sql.functions import col, trim

customers_silver = (
    customers_df

    # 1. Customer ID is a required key
    .filter(col('customer_id').isNotNull())

    # 2. Remove leading/trailing spaces
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))

    # 3. Replace missing customer name
    .fillna({"customer_name": "Unknown", "city": "Unknown", "state": "Unknown"})

    # 4. Remove duplicate customers
    .dropDuplicates(["customer_id"])
)

display(customers_silver)

## SCD TYPE1 SILVER_CUSTOMER

In [0]:
# customers_silver.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("ecommerce_catalog.silver.customers")

from delta.tables import DeltaTable
target_table = "ecommerce_catalog.silver.customers"

if spark.catalog.tableExists(target_table):

    target = DeltaTable.forName(spark, target_table)
    (
        target.alias("target")
        .merge(
            customers_silver.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    customers_silver.write \
        .format("delta") \
        .saveAsTable(target_table)

In [0]:
customers_df = spark.read.table("ecommerce_catalog.silver.customers")

display(customers_df)

# **PRODUCTS**

## PRODUCTS DATA QUALITY CHECKS

In [0]:

print("Total product rows:", products_df.count())

print("Duplicate product IDs:")
products_df.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null product IDs:")
products_df.filter(col("product_id").isNull()).show()

print("Null prices:")
products_df.filter(col("price").isNull()).show()

print("Invalid prices (<= 0):")
products_df.filter(col("price") <= 0).show()

In [0]:
from pyspark.sql.functions import col, trim

products_silver = (
    products_df

    # Required key
    .filter(col("product_id").isNotNull())

    # Standardize text columns
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))

    # Valid price
    .filter(col("price").isNotNull())
    .filter(col("price") > 0)
    .fillna({
        "product_name": "Unknown",
        "category": "Unknown"
    })

    # Remove duplicate products
    .dropDuplicates(["product_id"])
)

display(products_silver)

### SCD TYPE1 SILVER_PRODUCTS

In [0]:
# products_silver.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("ecommerce_catalog.silver.products")

target_table = "ecommerce_catalog.silver.products"

if spark.catalog.tableExists(target_table):

    target = DeltaTable.forName(spark, target_table)
    (
        target.alias("target")
        .merge(
            products_silver.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    products_silver.write \
        .format("delta") \
        .saveAsTable(target_table)

In [0]:
%sql
select * from ecommerce_catalog.silver.products

# ORDERS

In [0]:
display(orders_df)

## ORDER DATA QUALITY CHECK

In [0]:
from pyspark.sql.functions import col

print("Total order rows:", orders_df.count())

print("Duplicate order IDs:")
orders_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null order IDs:")
orders_df.filter(col("order_id").isNull()).show()

print("Null customer IDs:")
orders_df.filter(col("customer_id").isNull()).show()

print("Null product IDs:")
orders_df.filter(col("product_id").isNull()).show()

print("Invalid quantities:")
orders_df.filter(
    col("quantity").isNull() | (col("quantity") <= 0)
).show()

## ORDER DATA CLEANING

In [0]:
orders_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, to_date

orders_silver = (
    orders_df

    # Required order key
    .filter(col("order_id").isNotNull())

    # Required foreign keys
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())

    # Standardize text
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product_id", trim(col("product_id")))

    # Convert order_date to proper DATE
    .withColumn("order_date", to_date(col("order_date")))
    .filter(col("order_date").isNotNull())

    # Valid quantity
    .filter(col("quantity").isNotNull())
    .filter(col("quantity") > 0)

    # Remove duplicate orders
    .dropDuplicates(["order_id"])
)

display(orders_silver)

In [0]:
orders_silver.printSchema()

In [0]:
# orders_silver.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("ecommerce_catalog.silver.orders")

In [0]:
orders_silver.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()


orders_silver.filter(
    col("customer_id").isNull() |
    col("product_id").isNull()
).show()


orders_silver.filter(
    col("quantity").isNull() |
    (col("quantity") <= 0)
).show()

In [0]:
customers_df = spark.read.table("ecommerce_catalog.silver.customers")
order_df = spark.read.table("ecommerce_catalog.silver.orders")
products_df = spark.read.table("ecommerce_catalog.silver.products")


#customers_df.show()
display(customers_df)
display(products_df)
display(order_df)


In [0]:
unmatched_customers = (
    orders_silver
    .join(
        customers_silver,
        orders_silver.customer_id == customers_silver.customer_id,
        "left_anti"
    )
)

display(unmatched_customers)

In [0]:
unmatched_products = (
    orders_silver
    .join(
        products_silver,
        orders_silver.product_id == products_silver.product_id,
        "left_anti"
    )
)

display(unmatched_products)

In [0]:
valid_orders = (
    orders_silver
    .join(
        customers_silver.select("customer_id"),
        on="customer_id",
        how="inner"
    )
     .join(
         products_silver.select("product_id"),
         on="product_id",
         how="inner"
     )
)

display(valid_orders)

In [0]:
# valid_orders.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("ecommerce_catalog.silver.orders")


target_table = "ecommerce_catalog.silver.orders"

if spark.catalog.tableExists(target_table):

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("target")
        .merge(
            valid_orders.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    valid_orders.write \
        .format("delta") \
        .saveAsTable(target_table)


In [0]:
%sql
select * from ecommerce_catalog.silver.orders

In [0]:
customers_df = spark.read.table("ecommerce_catalog.silver.customers")
order_df = spark.read.table("ecommerce_catalog.silver.orders")
products_df = spark.read.table("ecommerce_catalog.silver.products")


#customers_df.show()
display(customers_df)
display(products_df)
display(order_df)